In [ ]:
# !pip install --quiet openai tiktoken

# LLMOPs- deploy

Latency, throuhput, error rates, observability: 

PII

cost-

Governance-blue (stable)/green(new candidate)

## Multi step LLM Pipeline: rag



In [1]:
import os
import re
import json
import uuid
import time
import asyncio
from typing import Dict, List, Tuple, Any
from dataclasses import dataclass, field
from datetime import datetime
from collections import deque

from openai import OpenAI  # openai>=1.0.0


In [2]:
LOG_DIR = "logs"
os.makedirs(LOG_DIR, exist_ok=True)

REQUEST_LOG = os.path.join(LOG_DIR, "requests.jsonl")
METRICS_LOG = os.path.join(LOG_DIR, "metrics.jsonl")
GOVERNANCE_LOG = os.path.join(LOG_DIR, "governance.jsonl")

# Dummy but realistic-style pricing (per 1M tokens)
MODEL_PRICING = {
    "gpt-4o-mini":  {"input_per_million": 0.15, "output_per_million": 0.60},
    "gpt-4o-small": {"input_per_million": 0.40, "output_per_million": 1.20},
    "gpt-4o":       {"input_per_million": 2.50, "output_per_million": 10.00},
}

# Blue/Green deployment simulation
BLUE_MODEL = "gpt-4o-mini"   # current stable
GREEN_MODEL = "gpt-4o"        # candidate upgraded model


In [3]:
def now_iso() -> str:
    return datetime.utcnow().isoformat() + "Z"


def append_jsonl(path: str, record: Dict[str, Any]) -> None:
    """Append a dict as one JSON line to the given file."""
    rec = dict(record)
    rec.setdefault("timestamp", now_iso())
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")


class StepTimer:
    """Context manager for measuring step latency in ms."""
    def __init__(self, step_name: str):
        self.step_name = step_name
        self.start = None
        self.latency_ms = None

    def __enter__(self):
        self.start = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc, tb):
        end = time.perf_counter()
        self.latency_ms = (end - self.start) * 1000.0 #30ms


In [4]:
#PII Masking
EMAIL_RE = re.compile(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}")
PHONE_RE = re.compile(r"\b(\+?\d[\d\s-]{7,15})\b")
AADHAAR_RE = re.compile(r"\b\d{4}\s?\d{4}\s?\d{4}\b")  # naive
PAN_RE = re.compile(r"\b[A-Z]{5}\d{4}[A-Z]\b")


def _mask_pattern(text: str, pattern: re.Pattern, label: str) -> str:
    return pattern.sub(f"<{label}_REDACTED>", text)


def mask_pii(text: str) -> Tuple[str, bool]:
    """Mask common PII. Returns (masked_text, was_modified)."""
    original = text
    text = _mask_pattern(text, EMAIL_RE, "EMAIL")
    text = _mask_pattern(text, PHONE_RE, "PHONE")
    text = _mask_pattern(text, AADHAAR_RE, "AADHAAR")
    text = _mask_pattern(text, PAN_RE, "PAN")
    return text, (text != original)


In [8]:
def estimate_cost(model: str, input_tokens: int, output_tokens: int) -> float:
    pricing = MODEL_PRICING.get(model)
    if not pricing:
        return 0.0
    in_cost = (input_tokens / 1_000_000) * pricing["input_per_million"]
    out_cost = (output_tokens / 1_000_000) * pricing["output_per_million"]
    return round(in_cost + out_cost, 8)

'''model-gpt-4o-mini
in= 2000
out=800
in_cost=0.15
out_cost=0.6'''

2000/(0.15*1000000)+800/(0.60*1000000)

0.014666666666666668

In [ ]:
# print('hello')

In [9]:
# programming, x12=oov=1
# programe ing
# tokenizer=vocab
# king he
# i like this product !

In [17]:
@dataclass
class StepMetrics:
    step_name: str #retriver, llm_draft, llm_refine
    latency_ms: float #
    input_tokens: int
    output_tokens: int
    model: str
    cost: float


@dataclass
class RequestTrace:
    request_id: str
    user_id: str
    total_latency_ms: float = 0.0
    steps: List[StepMetrics] = field(default_factory=list)
    region: str = "India"
    gdpr_safe: bool = True

    def to_dict(self) -> Dict[str, Any]:
        return {
            "request_id": self.request_id,
            "user_id": self.user_id,
            "total_latency_ms": self.total_latency_ms,
            "region": self.region,
            "gdpr_safe": self.gdpr_safe,
            "steps": [
                {
                    "step_name": s.step_name,
                    "latency_ms": s.latency_ms,
                    "input_tokens": s.input_tokens,
                    "output_tokens": s.output_tokens,
                    "model": s.model,
                    "cost": s.cost,
                }
                for s in self.steps
            ],
        }


class MetricsLogger:
    @staticmethod
    def log_trace(trace: RequestTrace) -> None:
        append_jsonl(METRICS_LOG, trace.to_dict())


In [18]:
@dataclass
class ModelStats:
    latency_ms: float
    cost: float
    quality_score: float  # e.g., RAGAS-like score (here simulated)


class GovernanceManager:
    def __init__(self, window_size: int = 20, green_fraction: float = 0.05):
        self.blue_model = BLUE_MODEL
        self.green_model = GREEN_MODEL
        self.green_stats: deque[ModelStats] = deque(maxlen=window_size)
        self.green_fraction = green_fraction  # ~5% traffic by default

    def choose_route(self, request_index: int) -> str:
        """Return 'blue' or 'green'. Very naive routing: every Nth request is green."""
        if self.green_fraction <= 0:#0.05
            return "blue"
        n = int(1 / self.green_fraction)
        if n == 0:
            return "green"
        if request_index % n == 0:
            return "green"
        return "blue"

    def record_result(self, route: str, stats: ModelStats):
        if route == "green":
            self.green_stats.append(stats)

    def should_rollback(self) -> bool:
        """Check rolling window of green stats and decide if rollback is needed."""
        if len(self.green_stats) < 5:  # need enough data
            return False

        avg_latency = sum(s.latency_ms for s in self.green_stats) / len(self.green_stats)
        avg_cost = sum(s.cost for s in self.green_stats) / len(self.green_stats)
        avg_quality = sum(s.quality_score for s in self.green_stats) / len(self.green_stats)

        too_slow = avg_latency > 2000  # 2 seconds
        too_expensive = avg_cost > 0.10
        too_low_quality = avg_quality < 0.7

        if too_slow or too_expensive or too_low_quality:
            append_jsonl(
                GOVERNANCE_LOG,
                {
                    "event": "rollback",
                    "from": self.green_model,
                    "to": self.blue_model,
                    "reason": {
                        "too_slow": too_slow,
                        "too_expensive": too_expensive,
                        "too_low_quality": too_low_quality,
                    },
                },
            )
            return True

        return False


In [19]:
DOCS = [
    {
        "id": "doc1",
        "text": "RAG stands for Retrieval-Augmented Generation. It combines a retriever and a generator model.",
    },
    {
        "id": "doc2",
        "text": "Monitoring LLM apps involves tracking latency, throughput, token usage, and error rates.",
    },
    {
        "id": "doc3",
        "text": "Cost management for LLMs requires choosing the right model size, caching responses, and limiting tokens.",
    },
    {
        "id": "doc4",
        "text": "Blue/green deployments allow you to test a new model version (green) while keeping the old one (blue) as fallback.",
    },
]


def simple_retriever(query: str, k: int = 2) -> List[Dict[str, Any]]:
    """Very naive retriever based on word overlap."""
    q_tokens = set(query.lower().split())
    scores = []
    for d in DOCS:
        d_tokens = set(d["text"].lower().split())
        score = len(q_tokens & d_tokens)
        scores.append((score, d))
    scores.sort(key=lambda x: x[0], reverse=True)
    return [d for score, d in scores[:k] if score > 0]


# Extremely simple token counter (for illustration only)
def rough_token_count(text: str) -> int:
    return max(1, len(text.split()))


# In-memory cache for answers
CACHE: Dict[str, str] = {}


In [20]:
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("Please set the OPENAI_API_KEY environment variable.")

client = OpenAI(api_key=api_key)

print("Client initialized. Ready to call the model.")


Client initialized. Ready to call the model.


In [21]:
def call_llm(model: str, system_prompt: str, user_prompt: str):
    """
    Works with both new OpenAI Responses API and Azure-compatible chat.completions API.
    Returns (content, input_tokens, output_tokens)
    """

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    resp = client.chat.completions.create(
        model=model,
        messages=messages,
    )

    # --- NEW OPENAI SDK (output[]) ---
    try:
        content = resp.output[0].content[0].text
    except Exception:
        pass
    else:
        usage = resp.usage
        return (
            content,
            usage.input_tokens,
            usage.output_tokens,
        )

    # --- LEGACY CHAT COMPLETION FORMAT (choices[].message) ---
    try:
        content = resp.choices[0].message["content"]
        usage = resp.usage
        return (
            content,
            usage.prompt_tokens,
            usage.completion_tokens,
        )
    except Exception:
        pass

    # --- FALLBACK for any unknown format ---
    print("⚠️ Warning: Unknown response format:", resp)
    content = str(resp)
    return (
        content,
        rough_token_count(system_prompt + " " + user_prompt),
        rough_token_count(content),
    )


In [22]:
class RAGPipeline:
    def __init__(self):
        self.gov = GovernanceManager()
        self.request_counter = 0

    @staticmethod
    def _mask_user_id(user_id: str) -> str:
        if len(user_id) <= 3:
            return "***"
        return "***" + user_id[-3:]

    def run_query(self, user_id: str, query: str) -> Dict[str, Any]:
        self.request_counter += 1
        request_id = str(uuid.uuid4())

        # 1) PII masking
        masked_query, pii_modified = mask_pii(query)
        cache_key = masked_query.strip().lower()

        start_total = time.perf_counter()
        trace = RequestTrace(
            request_id=request_id,
            user_id=self._mask_user_id(user_id),
            gdpr_safe=pii_modified,
        )

        # 2) Choose route (blue/green)
        route = self.gov.choose_route(self.request_counter)
        base_model = BLUE_MODEL if route == "blue" else GREEN_MODEL

        # 3) Retrieval step
        with StepTimer("retriever") as t_retr:
            retrieved_docs = simple_retriever(masked_query, k=2)
        retr_in_tok = rough_token_count(masked_query)
        retr_out_tok = rough_token_count(" ".join(d["text"] for d in retrieved_docs))

        trace.steps.append(
            StepMetrics(
                step_name="retriever",
                latency_ms=t_retr.latency_ms,
                input_tokens=retr_in_tok,
                output_tokens=retr_out_tok,
                model="retriever",
                cost=0.0,
            )
        )

        # 4) Check cache before LLMs
        if cache_key in CACHE:
            answer = CACHE[cache_key]
            total_latency = (time.perf_counter() - start_total) * 1000
            trace.total_latency_ms = total_latency
            MetricsLogger.log_trace(trace)

            append_jsonl(
                REQUEST_LOG,
                {
                    "request_id": request_id,
                    "user_id": trace.user_id,
                    "query": masked_query,
                    "answer": answer,
                    "route": route,
                    "model_used": base_model,
                    "from_cache": True,
                    "gdpr_safe": trace.gdpr_safe,
                },
            )

            # Update governance with cheap stats for cached response
            self.gov.record_result(
                route,
                ModelStats(
                    latency_ms=total_latency,
                    cost=0.0,
                    quality_score=0.8,  # placeholder
                ),
            )
            return {"answer": answer, "route": route, "from_cache": True}

        # 5) Draft answer
        context = "\n\n".join(d["text"] for d in retrieved_docs)
        draft_system = "You are a helpful assistant answering using the provided context."
        draft_user = f"Context:\n{context}\n\nQuestion: {masked_query}\n\nDraft a concise answer."
        with StepTimer("llm_draft") as t_draft:
            draft, in_tok_d, out_tok_d = call_llm(base_model, draft_system, draft_user)
        draft_cost = estimate_cost(base_model, in_tok_d, out_tok_d)

        trace.steps.append(
            StepMetrics(
                step_name="llm_draft",
                latency_ms=t_draft.latency_ms,
                input_tokens=in_tok_d,
                output_tokens=out_tok_d,
                model=base_model,
                cost=draft_cost,
            )
        )

        # 6) Refine answer
        refine_system = "You are a senior AI ops engineer. Refine the answer for clarity and correctness."
        refine_user = f"Question: {masked_query}\n\nDraft answer:\n{draft}\n\nReturn an improved final answer."
        with StepTimer("llm_refine") as t_ref:
            refined, in_tok_r, out_tok_r = call_llm(base_model, refine_system, refine_user)
        refine_cost = estimate_cost(base_model, in_tok_r, out_tok_r)

        trace.steps.append(
            StepMetrics(
                step_name="llm_refine",
                latency_ms=t_ref.latency_ms,
                input_tokens=in_tok_r,
                output_tokens=out_tok_r,
                model=base_model,
                cost=refine_cost,
            )
        )

        # 7) End-to-end latency
        trace.total_latency_ms = (time.perf_counter() - start_total) * 1000

        # 8) Log metrics & request
        total_cost = draft_cost + refine_cost
        MetricsLogger.log_trace(trace)

        append_jsonl(
            REQUEST_LOG,
            {
                "request_id": request_id,
                "user_id": trace.user_id,
                "query": masked_query,
                "retrieved_docs": [d["id"] for d in retrieved_docs],
                "draft_answer": draft,
                "final_answer": refined,
                "route": route,
                "model_used": base_model,
                "total_cost": total_cost,
                "gdpr_safe": trace.gdpr_safe,
                "from_cache": False,
            },
        )

        # 9) Update governance
        self.gov.record_result(
            route,
            ModelStats(
                latency_ms=trace.total_latency_ms,
                cost=total_cost,
                quality_score=0.8,  # TODO: plug in RAGAS-style metric here
            ),
        )
        _ = self.gov.should_rollback()  # if True, logs rollback event

        # 10) Cache & return
        CACHE[cache_key] = refined
        return {"answer": refined, "route": route, "from_cache": False}


In [23]:
pipeline = RAGPipeline()

query = "What is blue/green deployment in the context of LLM services?"
result = pipeline.run_query(user_id="user123", query=query)

print("Route:", result["route"])
print("From cache:", result["from_cache"])
print("\nAnswer:\n", result["answer"][:500])


⚠️ Warning: Unknown response format: ChatCompletion(id='chatcmpl-CiZxmkn669Cltb6dgJGbcQ0yYIXeM', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='In the context of LLM services, blue/green deployment is a strategy that enables the testing of a new model version (green) while maintaining the previous version (blue) as a fallback. This approach allows for seamless updates and ensures minimal downtime, as traffic can be gradually directed to the new version while monitoring its performance, thus allowing for quick rollbacks if issues arise.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1764740710, model='gpt-4o-mini-2024-07-18', object='chat.completion', service_tier='default', system_fingerprint='fp_b547601dbd', usage=CompletionUsage(completion_tokens=75, prompt_tokens=88, total_tokens=163, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audi

In [24]:
QUERIES = [
    "What is RAG and why is it useful?",
    "How can I monitor latency and throughput in LLM apps?",
    "Explain blue/green deployment for AI services.",
    "How can I reduce token costs for LLM usage?",
    "What does observability mean in LLMOps?",
]


async def run_single(pipeline: RAGPipeline, user_id: str, query: str):
    loop = asyncio.get_running_loop()
    # Run the blocking pipeline in a thread to avoid blocking event loop
    return await loop.run_in_executor(None, pipeline.run_query, user_id, query)


async def run_concurrent_requests(n: int = 10):
    pipeline = RAGPipeline()
    tasks = []
    for i in range(n):
        q = QUERIES[i % len(QUERIES)]
        user_id = f"user{i:03d}"
        tasks.append(asyncio.create_task(run_single(pipeline, user_id, q)))
    results = await asyncio.gather(*tasks)
    for i, r in enumerate(results):
        print(f"Request {i+1}: route={r['route']}, from_cache={r['from_cache']}")
        print(f"Answer snippet: {r['answer'][:120]}...\n")


# Run the load test
await run_concurrent_requests(10)


⚠️ Warning: Unknown response format: ChatCompletion(id='chatcmpl-CiZzDQZvpzhFxtA6xc7YbluQYobJe', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='RAG, or Retrieval-Augmented Generation, is a method that integrates a retriever model to source relevant information and a generator model to produce coherent responses. This approach is useful because it enhances the quality and relevance of generated text by allowing the model to access external knowledge, leading to more informed and contextually accurate outputs.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1764740799, model='gpt-4o-mini-2024-07-18', object='chat.completion', service_tier='default', system_fingerprint='fp_b547601dbd', usage=CompletionUsage(completion_tokens=64, prompt_tokens=77, total_tokens=141, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens

In [ ]:
def tail_file(path: str, n: int = 5):
    if not os.path.exists(path):
        print(f"{path} does not exist yet.")
        return
    with open(path, "r", encoding="utf-8") as f:
        lines = f.readlines()[-n:]
    print(f"Last {len(lines)} lines from {path}:\n")
    for line in lines:
        print(line.strip())


tail_file(REQUEST_LOG, n=5)
print("\n" + "="*80 + "\n")
tail_file(METRICS_LOG, n=5)
print("\n" + "="*80 + "\n")
tail_file(GOVERNANCE_LOG, n=5)
